# 02d | Strictly resume expanded pretraining from 610 to 1,220 updates

130U-FinLLM | 130U Financial Language Model

Complete 02c, confirming 610 cumulative expansion updates, retention before/after
and a final `verified: true` backup. This notebook uses the same expanded general-training data and original 8K vocabulary,
restoring weights, optimizer, learning-rate schedule, RNG state and data cursor without reinitializing the model.
There is no need to repeat 01b or 02c.

By default verify cumulative update 610 to 611, then resume to 1,220. This adds 610 updates,
nominally about 9,994,240 token positions, bringing the expansion stage to about 19,988,480 positions. Actual effective tokens
are calculated from checkpoints and reports. Keep the original seed's approximately 4.91M tokens separate from this stage.
This extends the stopping budget while retaining the original 6,104-update learning-rate schedule; it does not complete the 100M-token budget.

Select Runtime > Change runtime type > T4 GPU and run cells in order. Do not launch a second training process concurrently.
After disconnection, reconnect and check the original process status. To recover, rerun from the first cell,
without deleting checkpoints, disabling identity checks or switching to random initialization to bypass errors.

In [ ]:
import os, sys, json, shutil, subprocess, importlib
from pathlib import Path

REPOSITORY = "https://github.com/130U/130U-FinLLM.git"
REF = "main"  # Resolve and display the commit first; use the full 40-character SHA for reproduction.
PROJECT = Path("/content/130U-FinLLM")

def command(*args, cwd=None):
    # Explicitly relay subprocess output into the notebook's captured stdout.
    with subprocess.Popen([str(x) for x in args], cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, process.args)

if len(REF) == 40 and all(c in "0123456789abcdefABCDEF" for c in REF):
    REVISION = REF.lower()
else:
    refs = subprocess.check_output(["git", "ls-remote", REPOSITORY, REF], text=True).strip().splitlines()
    if len(refs) != 1:
        raise RuntimeError("REF does not resolve uniquely; enter the full commit SHA.")
    REVISION = refs[0].split()[0]

if not PROJECT.exists():
    command("git", "clone", "--no-checkout", REPOSITORY, PROJECT)
else:
    remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=PROJECT, text=True).strip()
    if remote.rstrip("/").removesuffix(".git") != REPOSITORY.removesuffix(".git"):
        raise RuntimeError("/content/130U-FinLLM belongs to another repository; choose a new directory.")
command("git", "fetch", "--depth", "1", "origin", REVISION, cwd=PROJECT)
command("git", "checkout", "--detach", REVISION, cwd=PROJECT)
os.chdir(PROJECT)
print("Pinned code revision:", REVISION)

# Keep the bundled Colab CUDA / PyTorch; install only the other project dependencies.
import torch
from importlib import metadata
from packaging.requirements import Requirement

def tokenizer_install_requirements():
    # Colab can include Transformers even though this project does not use it.
    # Pass its active tokenizer bounds to pip so an old runtime can retain 0.22,
    # while a current runtime (or an already broken one) can use 0.23.
    requirements = ["tokenizers>=0.21,<0.24"]
    try:
        existing = metadata.requires("transformers") or []
    except metadata.PackageNotFoundError:
        existing = []
    for value in existing:
        requirement = Requirement(value)
        if requirement.name.lower().replace("_", "-") == "tokenizers":
            if requirement.marker is None or requirement.marker.evaluate({"extra": ""}):
                requirements.append(str(requirement))
    return list(dict.fromkeys(requirements))

def check_tokenizer_compatibility(requirements):
    installed = metadata.version("tokenizers")
    incompatible = [value for value in requirements
                    if installed not in Requirement(value).specifier]
    if incompatible:
        raise RuntimeError(f"tokenizers {installed} is incompatible with the existing environment:{incompatible}")
    print("tokenizers compatibility check passed:", installed)

def check_project_dependencies():
    # Read the refreshed installed metadata, not the previous editable checkout.
    problems = []
    for value in metadata.requires("finllm-130u") or []:
        requirement = Requirement(value)
        if requirement.marker is not None and not requirement.marker.evaluate({"extra": ""}):
            continue
        try:
            installed = metadata.version(requirement.name)
        except metadata.PackageNotFoundError:
            problems.append(f"Missing {requirement.name}")
            continue
        if installed not in requirement.specifier:
            problems.append(f"{requirement.name}=={installed} does not satisfy {requirement}")
    if problems:
        raise RuntimeError("Project dependency check failed:" + "; ".join(problems))
    print("Installed project dependencies passed validation.")

def activate_project_in_kernel():
    # pip installs the editable .pth for NEW Python processes. A notebook kernel
    # that was already running does not process that .pth again automatically.
    source = (PROJECT / "src").resolve()
    expected = source / "finllm"
    if not (expected / "__init__.py").is_file():
        raise RuntimeError(f"Project Python source is missing:{expected}")
    sys.path[:] = [str(source)] + [entry for entry in sys.path if entry != str(source)]
    # Git may have changed code in the same directory since an earlier cell run.
    # Refresh only our modules; never reload Torch or its live CUDA state.
    for name in tuple(sys.modules):
        if name == "finllm" or name.startswith("finllm."):
            del sys.modules[name]
    importlib.invalidate_caches()
    package = importlib.import_module("finllm")
    runtime = importlib.import_module("finllm.runtime")
    for module, filename in ((package, "__init__.py"), (runtime, "runtime.py")):
        if Path(module.__file__).resolve() != expected / filename:
            raise RuntimeError(f"The notebook imported the wrong project location:{module.__file__}")
    if not callable(runtime.restore_seed_checkpoint):
        raise RuntimeError("The checkpoint restore interface is unavailable in this notebook.")
    print("Current notebook kernel import check passed:", runtime.__file__)

def check_fresh_process_import():
    # Verify the installed editable package separately, without injecting src
    # into PYTHONPATH. Both the current kernel and future CLI processes must work.
    probe = """import json, sys
from pathlib import Path
import finllm
import finllm.runtime as runtime
expected = Path(sys.argv[1]).resolve() / 'finllm'
for module, filename in ((finllm, '__init__.py'), (runtime, 'runtime.py')):
    if Path(module.__file__).resolve() != expected / filename:
        raise RuntimeError('Fresh process imported the wrong project: ' + str(module.__file__))
if not callable(runtime.restore_seed_checkpoint):
    raise RuntimeError('Fresh process checkpoint restore is unavailable')
print(json.dumps({'finllm': str(Path(finllm.__file__).resolve()), 'runtime': str(Path(runtime.__file__).resolve())}))
"""
    result = subprocess.check_output([sys.executable, "-c", probe, str(PROJECT / "src")],
                                     cwd=PROJECT, text=True)
    print("Fresh Python process import check passed:", result.strip())

if tuple(int(v) for v in torch.__version__.split("+")[0].split(".")[:2]) < (2, 5):
    raise RuntimeError("PyTorch >= 2.5 is required; select a newer Colab runtime.")
tokenizer_requirements = tokenizer_install_requirements()
print("Active tokenizers constraints:", tokenizer_requirements)
torch_version_before = metadata.version("torch")
torch_file_before = Path(torch.__file__).resolve()
# One resolver transaction replaces the previous finllm metadata AND its
# dependencies. Upgrading tokenizers first leaves old finllm bounds active and
# produces a misleading conflict warning even when the eventual install works.
# The exact installed Torch build is constrained, so pip keeps Colab's CUDA build.
command(sys.executable, "-m", "pip", "install", "-q", "-e", ".",
        "torch==" + torch_version_before, *tokenizer_requirements)
importlib.invalidate_caches()
if metadata.version("torch") != torch_version_before or Path(torch.__file__).resolve() != torch_file_before:
    raise RuntimeError("The PyTorch environment changed unexpectedly; preserve the output and stop training.")
check_tokenizer_compatibility(tokenizer_requirements)
check_project_dependencies()
activate_project_in_kernel()
check_fresh_process_import()
print("Notebook setup complete; PyTorch remains:", torch_version_before)

def finllm(*args):
    command(sys.executable, "-u", "-m", "finllm", *args, cwd=PROJECT)

def require(*paths):
    missing = [str(p) for p in paths if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("Stage inputs are missing; prepare them or restore them from Drive first:\n" + "\n".join(missing))

finllm("environment")
disk = shutil.disk_usage("/content")
print(f"Free disk space in /content:{disk.free / 2**30:.2f} GiB")
Path("runtime_revision.txt").write_text(REVISION + "\n")

In [ ]:
MOUNT_DRIVE = True
if not MOUNT_DRIVE:
    raise RuntimeError("This stage requires existing seed inputs and versioned backups; mount Drive first.")
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/130U-FinLLM")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive project directory:", DRIVE_ROOT)

In [ ]:
CONTINUATION_RETENTION_BEFORE_SUCCEEDED = False
CONTINUATION_PROBE_SUCCEEDED = False
CONTINUATION_TRAIN_SUCCEEDED = False
CONTINUATION_RETENTION_AFTER_SUCCEEDED = False
CONTINUATION_BACKED_UP = False
SEED_VERSION = "seed_v1"
EXPANSION_VERSION = "expansion_v1"
COMPLETED_UPDATES = 610
PROBE_UPDATES = 611
TARGET_UPDATES = 1220
if not torch.cuda.is_available():
    raise RuntimeError("Select a T4 GPU runtime; continued training does not start on CPU.")
from finllm.expansion_continuation import (
    prepare_expansion_continuation, run_expansion_continuation,
    retention_continuation_check, backup_expansion_continuation)
continuation = prepare_expansion_continuation(
    PROJECT, DRIVE_ROOT, EXPANSION_VERSION, SEED_VERSION,
    completed_updates=COMPLETED_UPDATES, probe_updates=PROBE_UPDATES,
    target_updates=TARGET_UPDATES)
print("Restored cumulative expansion updates:", continuation["restored_updates"])
print("Original 610-update artifacts preserved:", continuation["predecessor_archive"])
print("Strict-resume configuration:", json.dumps(continuation["config"], indent=2, ensure_ascii=False))
print("GPU:", torch.cuda.get_device_name(0))

Read the frozen 610-update checkpoint diagnostics on original seed general-validation positions, then at 1,220 cumulative updates
measure with the same tokenizer, data and positions. Retain expansion-validation loss separately. Both measure text prediction,
not bullish/bearish accuracy. Reuse original reports when all input identities match.

In [ ]:
CONTINUATION_RETENTION_BEFORE_SUCCEEDED = False
CONTINUATION_PROBE_SUCCEEDED = False
CONTINUATION_TRAIN_SUCCEEDED = False
CONTINUATION_RETENTION_AFTER_SUCCEEDED = False
CONTINUATION_BACKED_UP = False
continuation_retention_before = retention_continuation_check(continuation, "before")
print(json.dumps(continuation_retention_before, indent=2, ensure_ascii=False))
CONTINUATION_RETENTION_BEFORE_SUCCEEDED = True

In [ ]:
CONTINUATION_PROBE_SUCCEEDED = False
CONTINUATION_TRAIN_SUCCEEDED = False
CONTINUATION_RETENTION_AFTER_SUCCEEDED = False
CONTINUATION_BACKED_UP = False
if not globals().get("CONTINUATION_RETENTION_BEFORE_SUCCEEDED", False):
    raise RuntimeError("Complete original-validation diagnostics for the 610-update checkpoint first.")
continuation_probe = run_expansion_continuation(continuation, "probe", finllm)
print(json.dumps(continuation_probe, indent=2, ensure_ascii=False))
CONTINUATION_PROBE_SUCCEEDED = True

Continue to 1,220 expansion updates only after one-step recovery succeeds. For later same-version checkpoints within the target range, resume from the actual cursor
rather than rolling back to update 610. Save and sync every 10 updates; retain original 610-update artifacts, checkpoint identity and
retention reports in a separate archive. Record elapsed time and throughput per call; rerunning a completed stage preserves original measurements.
Incompatible data/tokenizer/configuration/runtime stops execution. The stopping budget may increase while schedule semantics remain unchanged.

In [ ]:
CONTINUATION_TRAIN_SUCCEEDED = False
CONTINUATION_RETENTION_AFTER_SUCCEEDED = False
CONTINUATION_BACKED_UP = False
if not globals().get("CONTINUATION_RETENTION_BEFORE_SUCCEEDED", False) or not globals().get("CONTINUATION_PROBE_SUCCEEDED", False):
    raise RuntimeError("Complete original-validation diagnostics and one-step strict-recovery verification first.")
continuation_result = run_expansion_continuation(continuation, "continue", finllm)
print(json.dumps(continuation_result, indent=2, ensure_ascii=False))
CONTINUATION_TRAIN_SUCCEEDED = True

In [ ]:
CONTINUATION_RETENTION_AFTER_SUCCEEDED = False
CONTINUATION_BACKED_UP = False
if not globals().get("CONTINUATION_TRAIN_SUCCEEDED", False):
    raise RuntimeError("This continuation segment has not completed successfully; keep existing backups.")
continuation_retention_after = retention_continuation_check(continuation, "after")
print(json.dumps(continuation_retention_after, indent=2, ensure_ascii=False))
CONTINUATION_RETENTION_AFTER_SUCCEEDED = True

In [ ]:
CONTINUATION_BACKED_UP = False
if not all(globals().get(flag, False) for flag in (
    "CONTINUATION_RETENTION_BEFORE_SUCCEEDED", "CONTINUATION_PROBE_SUCCEEDED",
    "CONTINUATION_TRAIN_SUCCEEDED", "CONTINUATION_RETENTION_AFTER_SUCCEEDED")):
    raise RuntimeError("Diagnostics, one-step verification and continued training have not all succeeded; do not report a complete stage backup.")
continuation_backup = backup_expansion_continuation(continuation)
print(json.dumps(continuation_backup, indent=2, ensure_ascii=False))
shutil.copy2(PROJECT / "runtime_revision.txt", continuation["continuation_drive"] / "runtime_revision.txt")
CONTINUATION_BACKED_UP = True
print("Expanded general continuation and verified backup complete:", continuation["paths"]["drive"] / "run")
print("Segment reports and diagnostics directory:", continuation["continuation_drive"])
print("Original 610-update artifact archive:", continuation["predecessor_archive"])
print("Original seed data and 300-update artifacts preserved:", continuation["paths"]["seed_drive"])

Completion criteria: new one-step and training-segment reports, 1,220 cumulative updates, actual new/cumulative tokens,
retention before/after and a final `verified: true` backup. Keep these outputs to inform the next stage.
After entering this segment, use 02d to resume. Preserve the original 02c 610-update report and measurements in a separate archive.
This notebook continues general pretraining without starting financial-domain training, SFT or DPO; it does not evaluate asset stance or scoring ability.